# Shared design-conditioned transfer: offline reproduction

Run all cells with this directory present, from this directory or the repository root. Python with NumPy is sufficient. Every metric and bootstrap interval is recalculated from exported predictions and frozen per-chemical comparator errors; summary files are used only for verification. The original private Kaggle notebook is linked in README.md; this notebook requires no Kaggle credentials or internet access.

See SOURCES.md for attribution and data terms. The full fixed training command is shown in the final cell and runs only after explicitly enabling it.


In [ ]:
from pathlib import Path
import os, sys, json, csv, subprocess
candidates = [Path.cwd(), Path.cwd() / "benchmarks" / "screen_out_neural"]
bundle = next((p.resolve() for p in candidates if (p / "aggregate.py").is_file() and (p / "checksums.json").is_file()), None)
if bundle is None:
    raise FileNotFoundError("Open the notebook from the repository root or its screen_out_neural directory.")
out = Path(os.environ.get("SCREEN_NEURAL_OUTPUT", str(bundle / "recomputed"))).resolve()
subprocess.run([sys.executable, str(bundle / "aggregate.py"), "--out", str(out)], check=True)


In [ ]:
report = json.loads((out / "resources.json").read_text())
assert report["status"] == "passed"
assert report["drug_endpoint_rows"] == 326 and report["main_table_rows"] == 26
assert report["max_main_table_difference"] <= 2e-6
print((out / "main_table.md").read_text())
print(json.dumps(report, indent=2))


In [ ]:
# Optional full training: install requirements-train.txt first.
# Original experiment used a single T4. Full CPU training is also supported.
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run([sys.executable, str(bundle / "run_experiment.py"),
                    "--device", "auto", "--out", str(bundle / "trained")], check=True)
